# Análise Integrada dos Dados de Acidentes, Tipos de Veículos e Vítimas

Este notebook reúne e organiza de forma unificada as análises dos dados de acidentes, tipos de veículos e vítimas do município de São Paulo, utilizando a base RENAEST. 

## Instalação e Importação de Bibliotecas
Certifique-se de que as bibliotecas necessárias estão instaladas.

In [ ]:
%pip install pandas matplotlib seaborn numpy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

## Download do dataset
O dataset para análise pode ser baixado manualmente através do link: https://dados.transportes.gov.br/dataset/42e2320b-ea67-4fdc-896f-71363e043fc6/resource/5f33e23b-bdd8-48bd-b267-27532ba22912/download/renaest_dabertos_20250412.zip<br>
Deve ser salvo e descompactado na pasta dataset<br>
Ou se preferir, rode a célula abaixo para fazer isso automaticamente


In [8]:
import requests
import zipfile
import os

link = "https://dados.transportes.gov.br/dataset/42e2320b-ea67-4fdc-896f-71363e043fc6/resource/5f33e23b-bdd8-48bd-b267-27532ba22912/download/renaest_dabertos_20250412.zip"
zip_path = "dataset.zip"
extract_dir = "dataset"

# Baixar o arquivo zip
with requests.get(link, stream=True) as r:
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# Extrair o conteúdo
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

# Remover o arquivo zip
os.remove(zip_path)

# Listar arquivos extraídos
print(os.listdir(extract_dir))

['Acidentes_DadosAbertos_20250412.csv', 'Localidade_DadosAbertos_20250412.csv', 'TipoVeiculo_DadosAbertos_20250412.csv', 'Vitimas_DadosAbertos_20250412.csv']


## Carregamento dos Dados
Ajuste os caminhos conforme a estrutura do projeto.

In [ ]:
# Caminhos dos arquivos
acidentes_path = 'dataset/renaest_dabertos_20250412/Acidentes_DadosAbertos_20250412.csv'
tipoveic_path = 'dataset/renaest_dabertos_20250412/TipoVeiculo_DadosAbertos_20250412.csv'
vitimas_path = 'dataset/renaest_dabertos_20250412/Vitimas_DadosAbertos_20250412.csv'

# Carregamento
if os.path.exists(acidentes_path):
    df_acidentes = pd.read_csv(acidentes_path, sep=';', encoding='utf-8', low_memory=False)
else:
    raise FileNotFoundError(f'Arquivo não encontrado: {acidentes_path}')
if os.path.exists(tipoveic_path):
    df_veic = pd.read_csv(tipoveic_path, sep=';', encoding='utf-8', low_memory=False)
else:
    raise FileNotFoundError(f'Arquivo não encontrado: {tipoveic_path}')
if os.path.exists(vitimas_path):
    df_vitimas = pd.read_csv(vitimas_path, sep=';', encoding='utf-8', low_memory=False)
else:
    raise FileNotFoundError(f'Arquivo não encontrado: {vitimas_path}')

## Limpeza e Filtragem dos Dados
Padronização do processo para todos os datasets.

In [ ]:
def limpar_e_filtrar(df, municipio_col, ano_col, uf_col, codigo_ibge_col=None):
    df = df.dropna(axis=1, how='all').drop_duplicates()
    if codigo_ibge_col and codigo_ibge_col in df.columns:
        df = df[(df[ano_col] >= 2020) & (df[uf_col] == 'SP') & (df[codigo_ibge_col] == 3550308)]
    elif 'chv_localidade' in df.columns:
        df = df[(df[ano_col] >= 2020) & (df[uf_col] == 'SP') & (df['chv_localidade'].astype(str).str.startswith('SP3550308'))]
    else:
        df = df[(df[ano_col] >= 2020) & (df[uf_col] == 'SP')]
    return df.reset_index(drop=True)

df_acidentes = limpar_e_filtrar(df_acidentes, 'municipio', 'ano_acidente', 'uf_acidente', 'codigo_ibge')
df_veic = limpar_e_filtrar(df_veic, 'municipio', 'ano_acidente', 'uf_acidente')
df_vitimas = limpar_e_filtrar(df_vitimas, 'municipio', 'ano_acidente', 'uf_acidente')

## Remoção de Colunas Irrelevantes

In [ ]:
colunas_remover = [
    'chv_localidade', 'latitude_acidente', 'longitude_acidente', 'codigo_ibge',
    'uf_acidente', 'ano_acidente', 'mes_acidente', 'mes_ano_acidente',
    'bairro_acidente', 'cep_acidente'
]
for df in [df_acidentes, df_veic, df_vitimas]:
    for col in colunas_remover:
        if col in df.columns:
            df.drop(columns=col, inplace=True)

## Funções Auxiliares

In [ ]:
def contar_valores_unicos_todas_colunas(df):
    resultado = {}
    for col in df.columns:
        n_unicos = df[col].nunique()
        valores_unicos = df[col].unique()
        resultado[col] = {'quantidade': n_unicos, 'valores': valores_unicos}
    return resultado

def contar_desconhecido_ou_nao_informado(df):
    resultado = {}
    for col in df.columns:
        count = df[col].isin(['DESCONHECIDO', 'NAO INFORMADO']).sum()
        if count > 0:
            resultado[col] = count
    return resultado

## Integração dos Dados
Adiciona o(s) tipo(s) de veículo(s) ao dataset de acidentes.

In [ ]:
veic_por_acidente = df_veic.groupby('num_acidente')['tipo_veiculo'].apply(lambda x: ', '.join(sorted(x.unique()))).reset_index()
df_acidentes = df_acidentes.merge(veic_por_acidente, on='num_acidente', how='left')

## Análises Estatísticas Unificadas
A seguir, exemplos de análises integradas.

In [ ]:
# Distribuição dos tipos de acidente
tipos = df_acidentes['tp_acidente'].value_counts()
tipos.plot(kind='bar', figsize=(10,5))
plt.title('Distribuição dos Tipos de Acidente')
plt.xlabel('Tipo de Acidente')
plt.ylabel('Quantidade')
plt.show()

# Acidentes por dia da semana
dias = df_acidentes['dia_semana'].value_counts()
dias.plot(kind='bar', color='skyblue', figsize=(8,4))
plt.title('Acidentes por Dia da Semana')
plt.xlabel('Dia da Semana')
plt.ylabel('Quantidade')
plt.show()

In [ ]:
# Óbitos por tipo de acidente
obitos_por_tipo = df_acidentes.groupby('tp_acidente')['qtde_obitos'].sum().sort_values(ascending=False)
percentuais = (obitos_por_tipo / obitos_por_tipo.sum() * 100).round(1)
top2 = percentuais.head(2)
print(f'Em relação ao tipo de sinistro, {top2.index[0].lower()} foram responsáveis por {top2.iloc[0]}% das mortes, seguidos por {top2.index[1].lower()} ({top2.iloc[1]}%)')
obitos_por_tipo.plot(kind='bar', figsize=(10,5))
plt.title('Óbitos por Tipo de Acidente')
plt.xlabel('Tipo de Acidente')
plt.ylabel('Quantidade de Óbitos')
plt.show()

In [ ]:
# Acidentes por condição meteorológica
condicoes = df_acidentes['cond_meteorologica'].value_counts()
condicoes.plot(kind='bar', color='green', figsize=(8,4))
plt.title('Acidentes por Condição Meteorológica')
plt.xlabel('Condição Meteorológica')
plt.ylabel('Quantidade')
plt.show()

In [ ]:
# Proporção de acidentes com e sem óbitos
obitos = df_acidentes['qtde_obitos'].apply(lambda x: 'Com Óbito' if x > 0 else 'Sem Óbito').value_counts()
obitos.plot(kind='pie', autopct='%1.1f%%', startangle=90, colors=['red','lightgray'], figsize=(5,3))
plt.title('Proporção de Acidentes com e sem Óbitos')
plt.ylabel('')
plt.show()

In [ ]:
# Ranking de acidentes com óbito por tipo de veículo
ranking_veiculos = (df_acidentes[df_acidentes['qtde_obitos'] > 0]['tipo_veiculo'].value_counts().sort_values(ascending=False))
print('Ranking de acidentes com óbito por tipo de veículo:')
print(ranking_veiculos)
ranking_veiculos.plot(kind='bar', color='tomato', figsize=(10,5))
plt.title('Ranking de Acidentes com Óbito por Tipo de Veículo')
plt.xlabel('Tipo de Veículo')
plt.ylabel('Quantidade de Acidentes com Óbito')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Análises de Vítimas
Exemplo de análise demográfica das vítimas fatais.

In [ ]:
# Considera apenas vítimas fatais
vitimas_fatais = df_vitimas[df_vitimas['gravidade_lesao'] == 'OBITO']
# Óbitos por sexo
obitos_por_sexo = vitimas_fatais['genero'].value_counts()
total_obitos = obitos_por_sexo.sum()
percentuais = (obitos_por_sexo / total_obitos * 100).round(1)
masc = percentuais.get('MASCULINO', 0)
fem = percentuais.get('FEMININO', 0)
print(f
)

In [ ]:
# Óbitos por faixa etária
obitos_por_faixa = vitimas_fatais['faixa_idade'].value_counts().sort_values(ascending=False)
total_obitos = obitos_por_faixa.sum()
percentuais_faixa = (obitos_por_faixa / total_obitos * 100).round(1)
faixa_mais_atingida = percentuais_faixa.index[0]
percentual_mais_atingida = percentuais_faixa.iloc[0]
print(f
)
vitimas_fatais.groupby(['faixa_idade', 'genero']).size().unstack().plot(kind='bar', stacked=True, figsize=(10,5))
plt.title('Óbitos por Faixa Etária e Gênero')
plt.xlabel('Faixa Etária')
plt.ylabel('Quantidade de Óbitos')
plt.legend(title='Gênero')
plt.tight_layout()
plt.show()

In [ ]:
# Tendência anual de mortes
vitimas_fatais['ano'] = pd.to_datetime(vitimas_fatais['data_acidente']).dt.year
obitos_por_ano = vitimas_fatais['ano'].value_counts().sort_index()
obitos_por_ano.plot(marker='o', linestyle='-', label='Linha', figsize=(8,4))
obitos_por_ano.plot(kind='bar', alpha=0.5, color='orange', label='Barra')
plt.title('Óbitos por Ano')
plt.xlabel('Ano')
plt.ylabel('Quantidade de Óbitos')
plt.legend(['Linha', 'Barra'])
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição mensal dos óbitos
vitimas_fatais['mes'] = pd.to_datetime(vitimas_fatais['data_acidente']).dt.month
obitos_por_mes = vitimas_fatais['mes'].value_counts().sort_index()
obitos_por_mes.plot(kind='bar', color='skyblue', figsize=(8,4))
plt.title('Óbitos por Mês do Ano')
plt.xlabel('Mês')
plt.ylabel('Quantidade de Óbitos')
plt.xticks(ticks=range(0,12), labels=['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'], rotation=0)
plt.tight_layout()
plt.show()